# Session 8 — Pair Programming — **SOLUTIONS**
# 🔒 LECTURER COPY

In [1]:
import pandas as pd
import numpy as np

## Task 1 — Audit

In [2]:
customers = pd.read_csv('customers.csv')
transactions = pd.read_csv('transactions.csv')
print("Customers:", customers.shape)
print("Transactions:", transactions.shape)
print("\nMissing in customers:")
print(customers.isna().sum())
print("\nMissing in transactions:")
print(transactions.isna().sum())
print("\nDtypes (customers):")
print(customers.dtypes)
print("\nDtypes (transactions):")
print(transactions.dtypes)

Customers: (100, 6)
Transactions: (400, 5)

Missing in customers:
customer_id     0
name            0
email           0
signup_date     0
country         0
age            10
dtype: int64

Missing in transactions:
txn_id         0
customer_id    0
txn_date       0
category       0
amount         0
dtype: int64

Dtypes (customers):
customer_id      int64
name               str
email              str
signup_date        str
country            str
age            float64
dtype: object

Dtypes (transactions):
txn_id           int64
customer_id      int64
txn_date           str
category           str
amount         float64
dtype: object


**Findings:** `age` has ~10 missing; `signup_date` and `txn_date` are strings, not dates; `email` has trailing whitespace on some rows.

## Task 2 — Fix ages

In [3]:
# Decision: fill with median because we want to keep all customers in the report.
# If ages were missing for systematic reasons (e.g., older users skip the question), we'd reconsider.
customers['age'] = customers['age'].fillna(customers['age'].median())
print("Missing ages after fill:", customers['age'].isna().sum())

Missing ages after fill: 0


## Task 3 — Dates

In [4]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'], format='%d/%m/%Y')
transactions['txn_date'] = pd.to_datetime(transactions['txn_date'])
transactions['txn_month'] = transactions['txn_date'].dt.to_period('M')
print(customers['signup_date'].dtype, transactions['txn_date'].dtype)
print("Months covered:", transactions['txn_month'].unique()[:5], "...")

datetime64[us] datetime64[us]
Months covered: <PeriodArray>
['2024-01', '2024-02', '2024-03', '2024-04', '2024-05']
Length: 5, dtype: period[M] ...


## Task 4 — Strings

In [5]:
customers['email'] = customers['email'].str.strip().str.lower()
print("Unique countries:", customers['country'].unique())  # already clean here, but worth checking

Unique countries: <StringArray>
['CA', 'US', 'NZ', 'AU', 'UK']
Length: 5, dtype: str


## Task 5 — Join

In [6]:
merged = pd.merge(transactions, customers, on='customer_id', how='left')
print(f"Merged: {len(merged)} rows (should equal {len(transactions)})")

Merged: 400 rows (should equal 400)


## Task 6 — Orphans

In [7]:
orphans = merged[merged['name'].isna()]
print(f"Orphan transactions: {len(orphans)}")
print("Orphan customer_ids:", sorted(orphans['customer_id'].unique()))

Orphan transactions: 33
Orphan customer_ids: [np.int64(101), np.int64(102), np.int64(103), np.int64(104), np.int64(105), np.int64(106), np.int64(107), np.int64(108), np.int64(109)]


## Task 7 — Pivot

In [8]:
clean = merged.dropna(subset=['name'])  # exclude orphans
pivot = clean.pivot_table(
    index='country',
    columns='txn_month',
    values='amount',
    aggfunc='sum',
    fill_value=0
).round(2)
pivot

txn_month,2024-01,2024-02,2024-03,2024-04,2024-05
country,,,,,
AU,3691.72,4692.76,4391.31,3147.54,1560.97
CA,5842.31,4971.69,4065.17,3178.48,3290.69
NZ,7319.40,4187.17,6613.73,6509.07,2085.25
UK,2813.79,3083.89,3414.69,2810.05,536.25
US,3309.85,2032.28,2373.23,2407.47,1419.57


## Task 8 — Top spenders per country

In [9]:
(clean.groupby(['country', 'customer_id', 'name'])['amount'].sum()
   .reset_index()
   .sort_values(['country', 'amount'], ascending=[True, False])
   .groupby('country').head(2)
   .round(2)
)

,country,customer_id,name,amount
11,AU,37,Customer_37,1787.20
14,AU,47,Customer_47,1772.41
19,CA,1,Customer_1,2129.06
25,CA,29,Customer_29,1891.91
67,NZ,85,Customer_85,1882.60
63,NZ,73,Customer_73,1828.85
80,UK,87,Customer_87,2124.91
75,UK,42,Customer_42,1700.71
92,US,54,Customer_54,1239.15
97,US,74,Customer_74,1193.98


## Save

In [10]:
clean.to_csv('clean_merged.csv', index=False)
print(f"Saved {len(clean)} rows to clean_merged.csv")

Saved 367 rows to clean_merged.csv
